# Задание 1. Подготовка данных
## Что нужно сделать
* Подготовьте датасет. Для решения задачи будем использовать датасет `WikiText-2` — сборник текстов, собранный из статей на Wikipedia. На его основе нужно реализовать класс датасета на PyTorch. 
* в датасете все последовательности будут фиксированной длины (определённое количество токенов до <MASK> + <MASK> + определённое количество токенов после). Поэтому не нужно реализовывать `padding` и `masking`, а также кастомный метод `collate_fn`.
* нужно собрать новый список, который состоит из трёх частей:
  * Левая часть: срез из списка `token_ids`, взятый перед индексом i, длиной `seq_len // 2`.
  * Маска: список из одного элемента `tokenizer.mask_token_id`.
  * Правая часть: срез из списка `token_ids`, взятый сразу после индекса i, длиной `seq_len // 2`.
* Для токенизации строки используйте `tokenizer.encode`.
* Контекст для токена на i-ой позиции можно собрать так: `context = token_ids[max(0, i - seq_len//2): i] + [tokenizer.mask_token_id] + token_ids[i+1: i+1+seq_len//2]`.
* В качестве таргета возьмите token_ids[i].
* Размер датасета можно посчитать так: len(self.samples).
* Контекст и таргет можно получить так: x, y = self.samples[idx].

In [1]:
import torch
import torch.nn as nn
import re
import random
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import BertTokenizerFast
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
# Фиксируем seed для воспроизводимости
random.seed(42)
torch.manual_seed(42)

## первый раз грузим из интернета и делаем локальный сейв

In [3]:
import pandas as pd

do_not_download_dataset = True  

if do_not_download_dataset:
    # читаем датасет из файла сейва
    dataset = pd.read_csv('../data/wikitext_save.csv', sep=';').rename(columns={'0':'text'}).fillna('')
else:
    # загружаем датасет WikiText-2
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    # сохраняем загруженный датасет в файл
    pd.DataFrame(
        [dataset['text']]
    ).T.to_csv('../data/wikitext_save.csv', sep=';')

## preprocessing

In [ ]:
# длины последовательностей в датасете
# seq_len = 7 => 3 токена до <MASK> + токен <MASK> + 3 токена после
seq_len = 7

# удаляем слишком короткие тексты
texts = [line for line in dataset["text"] if len(line.split()) >= seq_len]

# функция для "чистки" текстов
def clean_string(text):
    # приведение к нижнему регистру
    text = text.lower()
    # удаление всего, кроме латинских букв, цифр и пробелов
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # удаление дублирующихся пробелов, удаление пробелов по краям
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# "чистим" тексты
cleaned_texts = list(map(clean_string, texts))

## train-val split (and text length cut)

In [ ]:
# для упрощения используем только max_texts_count текстов
max_texts_count = 7000

# разбиение на тренировочную и валидационную выборки
val_size = 0.05

train_texts, val_texts = train_test_split(cleaned_texts[:max_texts_count], test_size=val_size, random_state=42)


Train texts: 6650, Val texts: 350


## torch datasets and loaders

In [6]:
# класс датасета
class MaskedBertDataset(Dataset):
    def __init__(self, texts, tokenizer, seq_len=7):
        self.samples = []
        for line in texts:
            token_ids = tokenizer.encode(line, add_special_tokens=False, max_length=512, truncation=True)
            if len(token_ids) < seq_len:
                continue
            for i in range(1, len(token_ids) - 1):
                context = token_ids[max(0, i - seq_len//2): i] + [tokenizer.mask_token_id] + token_ids[i+1: i+1+seq_len//2]
                if len(context) < seq_len:
                    continue
                target = token_ids[i]
                self.samples.append((context, target))
           
    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)

In [7]:
# Загружаем BERT токенизатор
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


# тренировочный и валидационный датасеты
train_dataset = MaskedBertDataset(train_texts, tokenizer, seq_len=seq_len)
val_dataset = MaskedBertDataset(val_texts, tokenizer, seq_len=seq_len)


# даталоадеры
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

## check results of torch datasets and loaders preparation

In [9]:
print(f"Train texts: {len(train_texts)}, Val texts: {len(val_texts)}")
print(f"Train dataset size: {len(train_dataset)}, Val dataset size: {len(val_dataset)}")
print(f"Train loader size: {len(train_loader)}, Val loader size: {len(val_loader)}")

Train texts: 6650, Val texts: 350
Train dataset size: 592449, Val dataset size: 29617
Train loader size: 9258, Val loader size: 463


# Задание 2. Код модели
## Что нужно сделать
* реализовать конструктор и forward-проход модели;
* в рекуррентном блоке использовать двунаправленность;
* реализовать возможность выбора рекуррентного блока (RNN, GRU или LSTM) в зависимости от аргумента rnn_type, переданного в конструктор;
* реализовать возможность выбора метода агрегации скрытых состояний (конкатенация concat или сумма sum) в зависимости от аргумента combine, переданного в конструктор;
* после реализации модели, воспользовавшись методом count_parameters, для разных входных аргументов создать объект модели и посчитать количество её параметров.
* Эмбеддинг-слой должен быть с параметрами `vocab_size` и `hidden_dim`: `nn.Embedding(vocab_size, hidden_dim)`.
* Выбор рекуррентного блока можно реализовать так: 
  * `rnn_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]`
  * `self.rnn = rnn_cls(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)`
* При конкатенации размер скрытого состояния после агрегации должен быть hidden_dim * 2, при сумме — просто hidden_dim:
  * `out_dim = hidden_dim * 2 if combine == "concat" else hidden_dim`
* Позицию центрального <MASK> токена можно посчитать как `center = x.size(1) // 2`.
* Реализовать агрегацию скрытых состояний можно так:
  * `hidden_agg = hidden_forward + hidden_backward if self.combine == "sum" else torch.cat([hidden_forward, hidden_backward], dim=1)`

In [10]:
class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, rnn_type="GRU", combine="concat"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.combine = combine


        rnn_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)


        out_dim = hidden_dim * 2 if combine == "concat" else hidden_dim
        self.fc = nn.Linear(out_dim, vocab_size)


    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        center = x.size(1) // 2
        hidden_forward = out[:, center, :out.size(2)//2]
        hidden_backward = out[:, center, out.size(2)//2:]
        hidden_agg = hidden_forward + hidden_backward if self.combine == "sum" else torch.cat([hidden_forward, hidden_backward], dim=1)
        linear_out = self.fc(hidden_agg)
        return linear_out
    


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


vocab_size = tokenizer.vocab_size  
hidden_dim = 128

rnn_types = ["RNN", "GRU", "LSTM"]
combine_methods = ["sum", "concat"]


# Сравнение
print(f"{'RNN Type':<8} | {'Combine':<6} | {'Params':>10}")
print("-" * 35)
for rnn_type in rnn_types:
    for combine in combine_methods:
        model = BiRNNClassifier(vocab_size, hidden_dim, rnn_type, combine)
        param_count = count_parameters(model)
        print(f"{rnn_type:<8} | {combine:<6} | {param_count:>10,}")

RNN Type | Combine |     Params
-----------------------------------
RNN      | sum    |  7,910,202
RNN      | concat | 11,817,018
GRU      | sum    |  8,042,298
GRU      | concat | 11,949,114
LSTM     | sum    |  8,108,346
LSTM     | concat | 12,015,162


# Задание 3. Обучение модели
## Что нужно сделать
* функцию evaluate: подсчёт функции потерь и подсчёт accuracy на валидационном датасете;
* часть цикла обучения: 
  * обнуление градиентов оптимизатора, 
  * расчёт выходов модели, 
  * подсчёт функции потерь, 
  * расчёт и обновление градиентов.
* Выход модели считается так: model(x_batch).
* Предсказанные токены можно получить методом argmax:
  * `preds = torch.argmax(x_output, dim=1)`.
* Количество верно угаданных токенов можно посчитать, сравнив таргет с предсказанными токенами:
  * `correct += (preds == y_batch).sum().item()`.
* Количество элементов в батче можно получить методом size:
  * `total += y_batch.size(0)`.

In [ ]:

model = BiRNNClassifier(vocab_size, rnn_type="LSTM", combine="concat")
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    sum_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_output = model(x_batch)
            loss = criterion(x_output, y_batch)
            preds = torch.argmax(x_output, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            sum_loss += loss.item()
    return sum_loss / len(loader), correct / total

100%|██████████| 9258/9258 [10:56<00:00, 14.09it/s]


Epoch 1 | Train Loss: 6.276 | Val Loss: 5.708 | Val Acc: 0.220


100%|██████████| 9258/9258 [10:49<00:00, 14.25it/s]


Epoch 2 | Train Loss: 4.877 | Val Loss: 5.496 | Val Acc: 0.245


100%|██████████| 9258/9258 [10:04<00:00, 15.31it/s]


Epoch 3 | Train Loss: 4.119 | Val Loss: 5.551 | Val Acc: 0.253


In [ ]:
run_model_training_cycle = False

if run_model_training_cycle:
    # тут основной цикл обучения
    n_epochs = 3

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.
        for x_batch, y_batch in tqdm(train_loader):
            # Здесь была удалена лишняя строка присваивания
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss, val_acc = evaluate(model, val_loader)
    
        # Дописал вывод acc, так как строка была обрезана
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}")

    torch.save(model.state_dict(), '../models/model_07_weights.save')
else:
    # просто загружаем модель из сейва
    model = BiRNNClassifier(vocab_size, rnn_type="LSTM", combine="concat")
    model.load_state_dict(torch.load('../models/model_07_weights.save'))
    model.eval()  # Переводим модель в режим инференса